[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/06_ddp_vs_fsdp.ipynb)

# 6 · DDP vs FSDP2 on two GPUs
Week 15. **Needs 2 GPUs** — Kaggle's free "GPU T4 × 2" accelerator works. `gpu/dist_train.py` trains the same model with DDP
(full replica per GPU, gradients all-reduced) or FSDP2 (`fully_shard` per block: parameters all-gathered just before use, gradients
reduce-scattered). Compare peak memory per GPU and step time.

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

assert torch.cuda.device_count() >= 2, "needs two GPUs (Kaggle T4 x2)"

In [ ]:
!torchrun --nproc_per_node=2 gpu/dist_train.py --mode ddp  --d 1024 --layers 12 --batch 4 --seq 256

In [ ]:
!torchrun --nproc_per_node=2 gpu/dist_train.py --mode fsdp --d 1024 --layers 12 --batch 4 --seq 256

In [ ]:
# Find the largest model each mode can train (doubling layers until OOM)
for mode in ["ddp", "fsdp"]:
    for layers in [16, 24, 32, 48]:
        r = !torchrun --nproc_per_node=2 gpu/dist_train.py --mode {mode} --d 1536 --layers {layers} --steps 6 --batch 2 --seq 256 2>&1 | tail -4
        ok = any("rank 0" in line for line in r)
        print(mode, layers, "OK" if ok else "OOM / failed")
        if not ok: break

## Your turn
1. Explain the memory numbers with the 16 bytes/param rule and the world size.
2. Add activation checkpointing (`torch.utils.checkpoint`) per block: what does it do to memory and step time?
3. Why is FSDP's step slower at small sizes but able to train models DDP can't fit? Where does the extra communication come from?